# Laboratorium 2 — świat Wumpusa i wnioskowanie w ASP (Clingo)

## Problem

*Świat Wumpusa* (Russell & Norvig, *Artificial Intelligence: A Modern Approach*, rozdz. 7) to klasyczny przykład środowiska dla **agenta opartego na wiedzy**. Jaskinia jest prostokątną planszą pól $(x, y)$, $1 \le x \le W$, $1 \le y \le H$. Myśliwy zaczyna na polu wejściowym $(1,1)$, patrząc na wschód. W jaskini znajdują się:

* **jamy** — wejście do jamy oznacza śmierć,
* **wumpus** — potwór, który zjada myśliwego wchodzącego na jego pole (martwy wumpus jest już niegroźny),
* **złoto** — cel wyprawy.

Pole wejściowe jest zawsze bezpieczne. Agent **nie widzi** planszy — zna tylko jej rozmiar oraz to, co poczuje na polach, które odwiedził:

| bodziec | znaczenie |
|---|---|
| *Stench* (smród) | na polu sąsiednim (krawędzią) jest wumpus — także martwy |
| *Breeze* (powiew) | na polu sąsiednim jest jama |
| *Glitter* (blask) | na bieżącym polu leży złoto |
| *Bump* (uderzenie) | agent wszedł w ścianę |
| *Scream* (krzyk) | właśnie wystrzelona strzała zabiła wumpusa |

**Akcje:** `left`, `right` (obrót o 90°), `forward` (krok), `grab` (podniesienie złota), `shoot` (strzała leci prosto do ściany i zabija pierwszego napotkanego żywego wumpusa), `climb` (wyjście z jaskini, tylko na polu $(1,1)$).

**Punktacja (jak w AIMA):** $+1000$ za każde wyniesione złoto, $-1000$ za śmierć, $-10$ za strzałę, $-1$ za każdą akcję.

## Podejście

Agent zapisuje w bazie wiedzy wyłącznie obserwacje (odwiedzone pola, percepcje, strzały). Reguły świata zapisane są w programie ASP `wumpus_kb.lp`: każdy model stabilny to jeden możliwy układ ukrytej części jaskini zgodny z obserwacjami. Clingo liczy

* **konsekwencje ostrożne** (prawdziwe we wszystkich modelach) — to, co agent **wie** (np. „pole jest bezpieczne”, „tu na pewno jest jama”),
* **konsekwencje odważne** (prawdziwe w jakimś modelu) — to, co jest **możliwe**.

Na tej podstawie agent wybiera cel (zgodnie z priorytetem: podnieś złoto → wróć z całym złotem → idź na najbliższe pewnie bezpieczne pole → zastrzel ewentualnego wumpusa na polu bez jamy → [opcjonalnie] zaryzykuj → wyjdź z jaskini), a trasę wyznacza przeszukiwaniem wszerz po polach, o których wie, że są bezpieczne.

Pliki: `world.py` (środowisko), `inference.py` (baza wiedzy i wywołania Clingo), `agent.py` (strategia i planowanie), `simulate.py` (pętla gry i statystyki).

## Rozważane warianty

Funkcja `run_game(config, max_actions, risky, agent_knows_gold, verbose)` pozwala porównywać następujące warianty:

1. **Standardowy świat z AIMA** (`config = AIMA_WORLD`) — plansza 4×4 z rys. 7.2: jamy w $(3,1)$, $(3,3)$, $(4,4)$, wumpus w $(1,3)$, złoto w $(2,3)$.
2. **Losowe światy** (`generate_world(width, height, pit_prob, n_wumpus, n_gold, rng=...)`):
   * dowolny rozmiar planszy (także prostokątnej),
   * każde pole poza wejściem jest jamą niezależnie z prawdopodobieństwem `pit_prob` (domyślnie 0.2),
   * **wiele wumpusów** — agent ma tyle strzał, ilu jest wumpusów; wumpus może stać na jamie,
   * **wiele sztuk złota** — złoto leży tylko na polach bez jam.

   Losowy świat może być nierozwiązywalny (np. złoto otoczone jamami albo brak bezpiecznego ruchu już na starcie) — wtedy rozsądny agent po prostu wychodzi z jaskini.
3. **Znana / nieznana liczba sztuk złota** (`agent_knows_gold`) — jeśli agent wie, ile jest złota, wraca od razu po zebraniu wszystkiego; w przeciwnym razie eksploruje, dopóki istnieją pewnie bezpieczne pola.
4. **Agent ostrożny / ryzykujący** (`risky`) — agent ostrożny wchodzi tylko na pola, których bezpieczeństwo **wynika logicznie** z wiedzy; ryzykujący, gdy takich pól brak, wchodzi na najmniej podejrzane pole, które nie jest na pewno śmiertelne (zamiast od razu się wycofać).

Ten sam zestaw wariantów jest dostępny z linii poleceń: `python simulate.py --help`.



## Zadania do wykonania

1. Uruchom symulację i przyjrzyj się, jak agent ostrożny zachowuje się w standardowym świecie AIMA. Zmodyfikuj parametr max_actions, aby sprawdzić co się stanie gdy agentowi skończą się akcje.
2. Jaką maksymalną nagrodę da się uzyskać w tym środowisku (znając rozkład świata)?
3. Dokończ tryb ryzykowny działania agenta (miejsce do uzupełnienia oznaczone komentarzem TODO (RISKY) w pliku agent.py). W trybie ryzykownym agent ma wchodzić na najbliższe pole, na którym może potencjalnie znajdować się wumpus lub jama.
4. Napisz kod rozgrywający po 100 losowych gier (losowy świat) agenta ostrożnego i ryzykownego i obliczający średnią wartość nagrody. Który agent radzi sobie lepiej?
5. Dodaj wariant "zliczający" do trybu ryzykownego: agent estymuje prawdopodobieństwa wystąpienia ryzyka za pomocą metody danger_probabilities i wybiera pole, dla którego ryzyko jest najniższe. Powtórz eksperyment z losowymi światami i porównaj skuteczność agentów.

Poniższy kod uruchamia symulację w świecie AIMA z agentem ostrożnym.

In [1]:
from __future__ import annotations

import argparse
import random
import time
from dataclasses import dataclass

from agent import Agent
from world import AIMA_WORLD, COUNTING_WORLD, World, WorldConfig, generate_world
from simulate import run_game

#config = AIMA_WORLD
config = COUNTING_WORLD
verbose = True
max_actions = 100
risky = False
unknown_gold = False

run_game(config, max_actions, risky, not unknown_gold, verbose)

True world (P pit, W/w live/dead wumpus, G gold):
...  ...  ...  ... 
.W.  ...  ...  ... 
...  P..  ...  ... 
...> ...  ...  ..G 

Agent knowledge description:

        Each cell has 4 characters:
        * hunter position: one of "^", ">", "v", "<" or " " if the hunter is somewhere else
        * general status: "." -> visited, "s" -> provably safe, "?" unknown
        * pit status: "P" -> certain, "p" -> possible, "b" -> breeze felt here, " " -> away from frontier
        * wumpus status: "W" -> certainly alive, "X" -> certainly dead, "w" -> possibly alive, "~" -> stench felt here, " " -> away from frontier
        

--- goal: explore (2, 1)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?  | ?  | ?  
 .  |>s  | ?  | ?  
   1. forward  -> (2, 1) breeze
--- goal: explore (1, 2)
 ?  | ?  | ?  | ?  
 ?  | ?  | ?  | ?  
 s  | ?p | ?  | ?  
 .  |^.b | ?p | ?  
   2. left     -> (2, 1) breeze
   3. left     -> (2, 1) breeze
   4. forward  -> (1, 1) 
   5. right    -> (1, 1) 
   6. forward  

GameResult(score=-10, actions=10, gold=0, alive=True, climbed=True, seconds=0.004719663000287255)